In [ ]:
from datetime import datetime
from pathlib import Path

import pandas as pd
import numpy as np
import re

RAW_PATH      = Path('../data/raw/VSA_Dataset_20260316.xlsx')
SHEET         = 'Sheet1'
PROCESSED_DIR = Path('../data/processed')
OUT_PATH      = PROCESSED_DIR / 'vsa_data_cleaned.csv'

print("Last Updated:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

In [ ]:
vsa_data_raw = pd.read_excel(RAW_PATH, sheet_name=SHEET)
vsa_data     = vsa_data_raw.copy()

vsa_data["Created"]     = pd.to_datetime(vsa_data["Created"], errors="coerce")
vsa_data_raw["Created"] = pd.to_datetime(vsa_data_raw["Created"], errors="coerce")

print(f'Loaded: {vsa_data.shape[0]:,} rows × {vsa_data.shape[1]} columns')
print(f'Date range: {vsa_data["Created"].min()} – {vsa_data["Created"].max()}')


In [ ]:
vsa_data = vsa_data.replace({None: np.nan})

# Identifying Sparse Rows

In [ ]:
sparse_rows_filter = vsa_data.isna().mean(axis=1) >= 0.80
sparse_rows = vsa_data[sparse_rows_filter]

print(f'Sparse rows (≥80% null): {sparse_rows_filter.sum()}')
display(sparse_rows)

Two rows were identified where all columns except 'Date Created' were null. These rows are useless for analysis, so they were dropped.

In [ ]:
vsa_data = vsa_data[~sparse_rows_filter] # Drops sparse rows

# Identifying Duplicates

In [ ]:
# Find exact duplicates
exact_dupes = vsa_data[vsa_data.duplicated(keep=False)]

print(f"Found {exact_dupes.shape[0]} exact duplicate rows")

In [ ]:
# Diagnostic — inspect a sample row after sparse row removal
display(vsa_data.iloc[0].to_frame(name='value'))

In [ ]:
key_col = 'StudentID'

vsa_data[key_col] = vsa_data[key_col].astype(str).str.upper().str.strip()

partial_dupes = vsa_data[vsa_data.duplicated(subset=key_col, keep=False)]
print(f"Found {len(partial_dupes)} partial duplicate rows across {partial_dupes[key_col].nunique()} students")

In [ ]:
diff_summary = []

for sid, group in partial_dupes.groupby(key_col):
    non_key_cols  = [c for c in vsa_data.columns if c != key_col]
    differing_cols = [c for c in non_key_cols if group[c].nunique() > 1]
    if differing_cols:
        diff_summary.append({
            'StudentID'        : sid,
            'differing_columns': differing_cols,
            'row_indices'      : group.index.tolist(),
        })

diff_df = pd.DataFrame(diff_summary)
display(diff_df)

In [ ]:
n_before = len(vsa_data)
vsa_data = (
    vsa_data
    .sort_values('Created')
    .drop_duplicates(subset=key_col, keep='last')
    .reset_index(drop=True)
)
print(f'Records after deduplication: {len(vsa_data):,} (removed {n_before - len(vsa_data)})')

Handling Duplicate Applicants
* Some applicants submitted multiple records. These are identified as duplicates based on InstitutionName and Student ID
* Since the latest submission is assumed to contain the most complete information, only the most recent record was kept.
* Created was used as the timestamp to determine recency.

Questions
* Why did Student ID = 900944334 apply first under Masters then a week later under a Bachelors?

# Cleaning GPA

In [ ]:
GPA_COL   = "GPA"
valid_min = 0.0
valid_max = 4.0

gpa_clean_col = GPA_COL + "_clean"

# Convert to numeric, coercing text to NaN
vsa_data[gpa_clean_col] = pd.to_numeric(vsa_data[GPA_COL], errors="coerce")

# Log invalid text values before coercion
invalid_mask   = vsa_data[gpa_clean_col].isna() & vsa_data[GPA_COL].notna()
invalid_counts = vsa_data.loc[invalid_mask, GPA_COL].value_counts()

# Zero GPAs → NaN
zero_count = (vsa_data[gpa_clean_col] == 0).sum()
vsa_data.loc[vsa_data[gpa_clean_col] == 0, gpa_clean_col] = np.nan

# Out-of-range → NaN
out_of_range_mask  = (vsa_data[gpa_clean_col] < valid_min) | (vsa_data[gpa_clean_col] > valid_max)
out_of_range_count = out_of_range_mask.sum()
vsa_data.loc[out_of_range_mask, gpa_clean_col] = np.nan

print({
    "invalid_text_values" : invalid_counts.to_dict(),
    "zero_values_removed" : int(zero_count),
    "out_of_range_removed": int(out_of_range_count),
    "total_null_after"    : int(vsa_data[gpa_clean_col].isna().sum()),
})

In [ ]:
# Were initial raw NaN GPAs first-year students?
raw_gpa_null_ids = vsa_data_raw.loc[vsa_data_raw['GPA'].isna(), 'StudentID'].astype(str).str.upper().str.strip()
vsa_data[vsa_data['StudentID'].isin(raw_gpa_null_ids)]['StudentClass']

In [ ]:
vsa_data[GPA_COL] = vsa_data[gpa_clean_col]
vsa_data.drop(columns=[gpa_clean_col], inplace=True)

**GPA Cleaning Summary**
- Converted from string to numeric (`pd.to_numeric`, `errors='coerce'`)
- 1 invalid text value (`'No GPA yet'`) → NaN
- 12 zero values → NaN
- 0 out-of-range values removed
- 2 originally null GPAs belonged to 1st-year students (confirmed above)
- **Total NaN after cleaning: 15**

# Consolidating Institution Name and Manually Entered Institutions

In [ ]:
# To identify some initial obvious errors/variations in free-text institution entries
def share_first_word(df, col):
    first_word = df[col].str.strip().str.split().str[0].str.lower()

    families = (
        df.assign(first_word=first_word)
        .groupby("first_word")[col]
        .apply(lambda x: sorted(x.str.strip().unique()))
        .reset_index()
        .rename(columns={col: "variants"})
    )
    families = families[families["variants"].apply(len) > 1]
    return families[families["first_word"] != "university"]

share_first_word(vsa_data_raw, "Other_Institution")

In [ ]:
# Check for acronyms in free-text institution entries
vsa_data_raw[
    vsa_data_raw['Other Institution'].str.strip().str.isupper().fillna(False)
]['Other Institution']

## Workflow to Clean Institution Column
1. Normalize (strip, lowercase, collapse whitespace)
2. Apply explicit corrections (acronyms, known abbreviations, mispellings)
3. Restore proper casing
4. Flag anything still needing manual review
5. Output changes 

### Constants

Notes:

* MISSPELLINGS uses regex to fix typos at the character level (ex. "collage" -> "college"), before CORRECTIONS is applied. (some of the keys in corrections had multiple entries with mispellings)

* TYPO_PATTERN is used only for flagging; it has no effect on the data changes. 

In [ ]:
# New dataset uses underscore-separated column names
INPUT_COL    = "Other_Institution"
CLEAN_SUFFIX = "_clean"
clean_col    = INPUT_COL + CLEAN_SUFFIX   # "Other_Institution_clean"

# Words that stay lowercase unless they are the first word
LOWER_WORDS = {"the", "for", "at", "in", "of", "on", "to"}

# Explicit typos to search for (mispellings seen in data)
MISSPELLINGS = {
    r"\bcollage\b"          : "college",
    r"\bunivercity\b"       : "university",
    r"\bembry[\s-]riddle\b" : "embry-riddle",
}

# Keys are lowercase normalized forms, values are the canonical cased form.
CORRECTIONS = {

    # Acronyms
    "tamuct" : "Texas A&M University Central Texas",

    # Informal names/variations
    "penn state" : "Pennsylvania State University",
    "penn state university" : "Pennsylvania State University",
    "university of michigan ann arbor" : "University of Michigan",
    "university of michigan - ann arbor" : "University of Michigan",
    "embry-riddle aeronautical university prescott" : "Embry-Riddle Aeronautical University",
    "embry-riddle aeronautical university prescott campus" : "Embry-Riddle Aeronautical University",
    "rutgers newark university" : "Rutgers University-Newark",
    "university of maryland global campus" : "University of Maryland",
    "texas a&m corpus christi" : "Texas A&M University Corpus Christi",
    "texas a&m university- corpus christi" : "Texas A&M University Corpus Christi",
    "university of puerto rico-mayaguez campus" : "University of Puerto Rico Mayaguez",
    "the university of west florida" : "University of West Florida",
}

# Other potential typos to flag for manual review (applied after corrections)
TYPO_PATTERN = re.compile(
    r"\b(univeristy|universtiy|institue|insitute)\b",
    re.IGNORECASE
)

### Normalize (and Restore Caps) Functions

In [ ]:
def normalize(series: pd.Series) -> pd.Series:
    """Lowercase, strip, collapse spaced hyphens, remove commas/slashes."""
    s = (
        series
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"\s+-\s+", " ", regex=True)
        .str.replace(r"-\s+",      "-", regex=True) 
        .str.replace(r"\s+-",      "-", regex=True)  

        .str.replace(r"[,/]", "", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace("nan", pd.NA)
    )
    for pattern, replacement in MISSPELLINGS.items():
        s = s.str.replace(pattern, replacement, regex=True)
    return s

# For corrections dict processing
def _normalize_str(s: str) -> str:
    return normalize(pd.Series([s])).iloc[0]

# To restore proper capitalization later
def smart_title(s: str) -> str:
    """
    Capitalizes the first letter of each word except defined LOWER_WORDS, which stay lowercase unless they are
    the first word. Also capitalizes letters that immediately follow hyphens and ampersands.
    """

    words = s.split()
    result = []
    for i, word in enumerate(words):
        if i > 0 and word.lower() in LOWER_WORDS:
            result.append(word.lower())
        else:
            # Capitalize after spaces, hyphens, and ampersands
            word = re.sub(r"(^|[-&])([a-z])", lambda m: m.group(1) + m.group(2).upper(), word)
            result.append(word)
    return " ".join(result)

### Normalize and Apply Corrections

In [ ]:
vsa_data[clean_col] = normalize(vsa_data[INPUT_COL])

# Normalize keys, keep values as entered 
normalized_corrections = {_normalize_str(k): v for k, v in CORRECTIONS.items()}

# Applying corrections
vsa_data[clean_col] = vsa_data[clean_col].replace(normalized_corrections)

### Restore proper casing

In [ ]:
# This is applying proper casing to everything not in CORRECTIONS dict (could consider changing this to apply to CORRECTIONS)
vsa_data["institution_final"] = vsa_data[clean_col].astype(object).apply(
    lambda x: x if pd.isna(x)
    else x if x in normalized_corrections.values()
    else smart_title(x)
)

### Flag entries needing manual review

This section flags typos that could occur, entires that are all caps, and entires with 2+ letter all caps. Additional cases can be added.

In [ ]:
unique_entries = (
    vsa_data[[INPUT_COL, clean_col, "institution_final"]]
    .dropna(subset=[clean_col])
    .drop_duplicates(subset=clean_col)
    .copy()
)

unique_entries["flag_typo"] = unique_entries[clean_col].str.contains(
    TYPO_PATTERN.pattern, regex=True, na=False
)
unique_entries["flag_all_caps"] = unique_entries[INPUT_COL].apply(
    lambda x: str(x).strip().isupper() if pd.notna(x) else False
)
unique_entries["flag_acronym"] = unique_entries[INPUT_COL].apply(
    lambda x: bool(re.search(r"\b[A-Z]{2,}\b", str(x))) and not str(x).strip().isupper()
    if pd.notna(x) else False
)

flag_cols    = ["flag_typo", "flag_all_caps", "flag_acronym"]
review_table = (
    unique_entries[unique_entries[flag_cols].any(axis=1)]
    [[INPUT_COL, "institution_final"] + flag_cols]
    .sort_values(INPUT_COL)
)

print(f"Entries flagged for manual review: {len(review_table)}")
print(review_table.to_string(index=False))

### Changes DF

In [ ]:
changes_df = (
    vsa_data.loc[vsa_data[INPUT_COL].notna(), [INPUT_COL, "institution_final"]]
    .drop_duplicates(subset=INPUT_COL)
    .rename(columns={INPUT_COL: "original_entry", "institution_final": "converted_to"})
    .sort_values("original_entry")
    .reset_index(drop=True)
)

# To count changes that are different when before and after are normalized
changes_df["meaningfully_changed"] = (
    normalize(changes_df["original_entry"]) !=
    normalize(changes_df["converted_to"])
)

# To count changes that are different only because of casing (all changes not in 'meaningfully_changed')
changes_df["casing_only"] = (
    (changes_df["original_entry"] != changes_df["converted_to"]) &
    ~changes_df["meaningfully_changed"]
)

print(f"Total unique entries    : {len(changes_df)}")
print(f"Meaningfully changed   : {changes_df['meaningfully_changed'].sum()}")
print(f"Casing corrected only  : {changes_df['casing_only'].sum()}")
print(f"Unchanged              : {(~changes_df['meaningfully_changed'] & ~changes_df['casing_only']).sum()}")

In [ ]:
# Combine: use cleaned free-text (institution_final) where available,
# then fall back to InstitutionName dropdown.
# Sentinel values ("Other - My University is not listed") never reach Institution
# because Other Institution is always populated when the sentinel is selected.
vsa_data["Institution"] = vsa_data["institution_final"].fillna(vsa_data["InstitutionName"])

print(f'Null Institution after merge: {vsa_data["Institution"].isna().sum()}')

## Dropping helper columns and original columns

In [ ]:
# Preserve flag for whether free-text institution was used
vsa_data["other_institution_flag"] = vsa_data[INPUT_COL].notna()

# Drop helper and source columns (INPUT_COL = "Other Institution")
vsa_data = vsa_data.drop(columns=[
    INPUT_COL,
    clean_col,
    "institution_final",
    "InstitutionName",
], errors="ignore")

Check for obvious remaining errors

In [ ]:
share_first_word(vsa_data, "Institution")

**Notes:**

* A&M formatted differently in the 'InstitutionName' dropdown column ('A&M' and 'A & M')
* Decisions need to be made regarding when to include campus/regional names
* Need to ensure entries beginning with 'The' are correct 

# Consolidating Discipline and OtherMajor Columns

In [ ]:
print(f'OtherMajor non-null entries: {vsa_data["OtherMajor"].notna().sum()}')
share_first_word(vsa_data_raw, 'OtherMajor')

1. Normalize (lowercase, collapse whitespace, keeping slashes)
2. Apply explicit corrections
3. Restore proper casing
4. Output changes

Due to the wide variety of majors, only limited changes were made to achieve consistent formatting.

## Constants

In [ ]:
DISCIPLINE_CORRECTIONS = {
    "business analytics - mix of data science and statistics" : "Business Analytics",
    "criminal justiceintelligence analysis" : "Criminal Justice Intelligence Analysis"
}

MAJOR_INPUT_COL = "OtherMajor"
discipline_clean_col = MAJOR_INPUT_COL + CLEAN_SUFFIX

## Normalize and Apply Corrections

In [ ]:
MAJOR_INPUT_COL      = "OtherMajor"
discipline_clean_col = MAJOR_INPUT_COL + CLEAN_SUFFIX   # "OtherMajor_clean"

vsa_data[discipline_clean_col] = (
    vsa_data[MAJOR_INPUT_COL]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .replace("nan", pd.NA)
)

# Apply corrections
normalized_discipline_corrections = {
    _normalize_str(k): v.strip()
    for k, v in DISCIPLINE_CORRECTIONS.items()
}

vsa_data[discipline_clean_col] = vsa_data[discipline_clean_col].replace(normalized_discipline_corrections)

## Restore Casing

In [ ]:
# Restore casing
vsa_data["discipline_final"] = vsa_data[discipline_clean_col].astype(object).apply(
    lambda x: x if pd.isna(x)
    else x if x in normalized_discipline_corrections.values()
    else smart_title(x)
)

In [ ]:
# Combine: cleaned free-text (discipline_final) first, then Discipline dropdown.
# Same pattern as institution — OtherMajor is always filled when "Other" is selected.
vsa_data["Major"] = vsa_data["discipline_final"].fillna(vsa_data["Discipline"])

# Flag rows that used the free-text entry
vsa_data["other_major_flag"] = vsa_data[MAJOR_INPUT_COL].notna()

# Drop original and helper columns
vsa_data = vsa_data.drop(columns=[
    MAJOR_INPUT_COL,           # "OtherMajor"
    discipline_clean_col,      # "OtherMajor_clean"
    "Discipline_clean",
    "discipline_final",
    "Discipline",
], errors="ignore")

print(f'Null Major after merge: {vsa_data["Major"].isna().sum()}')

In [ ]:
share_first_word(vsa_data, 'Major')

# Saving to data/processed

In [ ]:
print(f'Original shape {vsa_data_raw.shape}')
print(f'Shape after cleaning {vsa_data.shape}')

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

vsa_data.to_csv(OUT_PATH, index=False)

print(f'Original shape:        {vsa_data_raw.shape[0]:,} rows × {vsa_data_raw.shape[1]} columns')
print(f'Cleaned shape:         {vsa_data.shape[0]:,} rows × {vsa_data.shape[1]} columns')
print(f'Saved to:              {OUT_PATH.resolve()}')